**Working with Data Frame**



In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("widgets").getOrCreate()

In [3]:
from google.colab import files, output

# Enable interactive widgets to Google Colab
output.enable_custom_widget_manager()

import pandas as pd
import ipywidgets as widgets
from IPython.display import display

# Upload the manufacturing CSV file
uploaded = files.upload()

file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("Number of records:", len(df))

display(df.head())

Saving manufacturing_production_extra_12k.csv to manufacturing_production_extra_12k.csv
Dataset loaded successfully!
Number of records: 12500


,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
0,PR0120001,2026-06-01,PL05,MC024,OP0191,Evening,PMP-210,Pumps,293,292,10,44.0,120.42,70.59,2.21,Pass,Yes,Anita
1,PR0120002,2026-09-22,PL01,MC001,OP0461,Night,MTR-100,Motors,1009,897,25,59.0,317.23,81.74,3.03,Pass,No,Anita
2,PR0120003,2026-05-24,PL03,MC012,OP0451,Evening,MTR-100,Motors,960,983,34,0.0,350.82,67.29,2.01,Pass,No,Deepa
3,PR0120004,2026-09-12,PL04,MC016,OP0237,Night,CTL-130,Controllers,310,320,15,14.0,142.34,59.70,3.48,Pass,No,Anita
4,PR0120005,2026-02-04,PL04,MC018,OP0072,Morning,VLV-220,Valves,957,979,22,14.0,296.70,62.52,2.33,Pass,No,Harini


In [4]:
# Function to create dropdown options
def create_options(column, all_label):
    values = sorted(df[column].dropna().astype(str).unique())

    return [(all_label, "__ALL__")] + [
        (value, value) for value in values
    ]


# 1. Plant dropdown
plant_dropdown = widgets.Dropdown(
    options=create_options("plant_id", "All Plants"),
    description="Plant:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="260px")
)


# 2. Product family dropdown
product_dropdown = widgets.Dropdown(
    options=create_options("product_family", "All Products"),
    description="Product Family:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="300px")
)


# 3. Quality status dropdown
quality_dropdown = widgets.Dropdown(
    options=create_options("quality_status", "All Statuses"),
    description="Quality:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="260px")
)


# 4. Slider for units produced
units = pd.to_numeric(df["units_produced"], errors="coerce").dropna()

units_slider = widgets.IntRangeSlider(
    value=(int(units.min()), int(units.max())),
    min=int(units.min()),
    max=int(units.max()),
    step=1,
    description="Units Produced:",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="90%")
)

In [5]:
from IPython.display import display
from google.colab import output

# Enable interactive widgets in Google Colab
output.enable_custom_widget_manager()

# Display all four widgets
display(widgets.VBox([
    plant_dropdown,
    product_dropdown,
    quality_dropdown,
    units_slider
]))

In [6]:
# Function to create dropdown option
def create_options(column, all_label):
    values = sorted(df[column].dropna().astype(str).unique())

    return [(all_label, "__ALL__")] + [
        (value, value) for value in values
    ]

# Shift dropdown
shift_dropdown = widgets.Dropdown(
    options=create_options("shift", "All Shifts"),
    description="Shift:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="260px")
)

In [7]:
from IPython.display import display
from google.colab import output

# Enable interactive widgets in Google Colab
output.enable_custom_widget_manager()

# Display the shift widget
display(widgets.VBox([shift_dropdown]))

In [8]:
# Function to filter and display data
def filter_data(shift):
    if shift == "_ALL_":
        filtered_df = df
    else:
        filtered_df = df[df["shift"].astype(str) == shift]

    display(filtered_df)

# Connect dropdown to filtering function
out = widgets.interactive_output(
    filter_data,
    {"shift": shift_dropdown}
)

# Display dropdown and filtered data
display(widgets.VBox([shift_dropdown, out]))

**Transformations and Actions**

In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, desc

spark = SparkSession.builder \
    .appName("LazyExecutionDemo") \
    .getOrCreate()

data = [
    (1, "Hyderabad", "Laptop", 2, 55000),
    (2, "Bangalore", "Mobile", 5, 20000),
    (3, "Chennai", "Tablet", 3, 15000),
    (4, "Hyderabad", "Mobile", 4, 22000),
    (5, "Mumbai", "Laptop", 1, 60000),
    (6, "Bangalore", "Tablet", 6, 12000),
    (7, "Chennai", "Laptop", 2, 58000),
    (8, "Mumbai", "Mobile", 8, 18000),
    (9, "Hyderabad", "Tablet", 5, 14000),
    (10, "Bangalore", "Laptop", 3, 52000),
    (11, "Mumbai", "Tablet", 7, 13000),
    (12, "Chennai", "Mobile", 6, 21000)
]

columns = ["sale_id", "city", "product", "quantity", "price"]

df = spark.createDataFrame(data, columns)

df.show()

+-------+---------+-------+--------+-----+
|sale_id|     city|product|quantity|price|
+-------+---------+-------+--------+-----+
|      1|Hyderabad| Laptop|       2|55000|
|      2|Bangalore| Mobile|       5|20000|
|      3|  Chennai| Tablet|       3|15000|
|      4|Hyderabad| Mobile|       4|22000|
|      5|   Mumbai| Laptop|       1|60000|
|      6|Bangalore| Tablet|       6|12000|
|      7|  Chennai| Laptop|       2|58000|
|      8|   Mumbai| Mobile|       8|18000|
|      9|Hyderabad| Tablet|       5|14000|
|     10|Bangalore| Laptop|       3|52000|
|     11|   Mumbai| Tablet|       7|13000|
|     12|  Chennai| Mobile|       6|21000|
+-------+---------+-------+--------+-----+



In [11]:
# Transformation 1
filtered_df = df.filter(col("quantity") > 3)

print("Transformation created")

# Transformation 2
selected_df = filtered_df.select("city", "product", "quantity")

print("Another transformation created")

# Action
selected_df.show()

Transformation created
Another transformation created
+---------+-------+--------+
|     city|product|quantity|
+---------+-------+--------+
|Bangalore| Mobile|       5|
|Hyderabad| Mobile|       4|
|Bangalore| Tablet|       6|
|   Mumbai| Mobile|       8|
|Hyderabad| Tablet|       5|
|   Mumbai| Tablet|       7|
|  Chennai| Mobile|       6|
+---------+-------+--------+



In [12]:
# Transformation 1: Filter sales
step1 = df.filter(col("quantity") >= 3)

# Transformation 2: Calculate revenue
step2 = step1.withColumn("revenue", col("quantity") * col("price"))

# Transformation 3: Group by city
step3 = step2.groupBy("city").agg(
    sum("revenue").alias("total_revenue")
)

# Transformation 4: Sort results
step4 = step3.orderBy(desc("total_revenue"))

print("All Transformations defined")

# Nothing above requires Spark to execute the query.

# Action: Display results
step4.show()

All Transformations defined
+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|Bangalore|       328000|
|   Mumbai|       235000|
|  Chennai|       171000|
|Hyderabad|       158000|
+---------+-------------+



In [13]:
# Define Transformations
result = df.filter(col("price") > 20000).select("city", "product", "price").orderBy(desc("price"))

# View Spark's execution plan
result.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [price#4L DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(price#4L DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=109]
      +- Project [city#1, product#2, price#4L]
         +- Filter (isnotnull(price#4L) AND (price#4L > 20000))
            +- Scan ExistingRDD[sale_id#0L,city#1,product#2,quantity#3L,price#4L]




In [14]:
result.show()

+---------+-------+-----+
|     city|product|price|
+---------+-------+-----+
|   Mumbai| Laptop|60000|
|  Chennai| Laptop|58000|
|Hyderabad| Laptop|55000|
|Bangalore| Laptop|52000|
|Hyderabad| Mobile|22000|
|  Chennai| Mobile|21000|
+---------+-------+-----+



In [15]:
high_value_sales = df.filter(col("price") > 20000)

# Action 1
high_value_sales.show()

# Action 2
print("Total records:", high_value_sales.count())

# Action 3
records = high_value_sales.collect()
print(records)

+-------+---------+-------+--------+-----+
|sale_id|     city|product|quantity|price|
+-------+---------+-------+--------+-----+
|      1|Hyderabad| Laptop|       2|55000|
|      4|Hyderabad| Mobile|       4|22000|
|      5|   Mumbai| Laptop|       1|60000|
|      7|  Chennai| Laptop|       2|58000|
|     10|Bangalore| Laptop|       3|52000|
|     12|  Chennai| Mobile|       6|21000|
+-------+---------+-------+--------+-----+

Total records: 6
[Row(sale_id=1, city='Hyderabad', product='Laptop', quantity=2, price=55000), Row(sale_id=4, city='Hyderabad', product='Mobile', quantity=4, price=22000), Row(sale_id=5, city='Mumbai', product='Laptop', quantity=1, price=60000), Row(sale_id=7, city='Chennai', product='Laptop', quantity=2, price=58000), Row(sale_id=10, city='Bangalore', product='Laptop', quantity=3, price=52000), Row(sale_id=12, city='Chennai', product='Mobile', quantity=6, price=21000)]


**Exercises - Transformations and Actions**

In [16]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("exercises").getOrCreate()

In [17]:
from google.colab import files, output

# Enable interactive widgets to Google Colab
output.enable_custom_widget_manager()

import pandas as pd
import ipywidgets as widgets
from IPython.display import display

# Upload the manufacturing CSV file
uploaded = files.upload()

file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("Number of records:", len(df))

display(df.head())

Saving lazy_execution_stores.csv to lazy_execution_stores.csv
Saving lazy_execution_sales_25000.csv to lazy_execution_sales_25000.csv
Saving lazy_execution_products.csv to lazy_execution_products.csv
Dataset loaded successfully!
Number of records: 8


,store_id,city,region,store_type
0,S01,Hyderabad,South,Mall
1,S02,Bengaluru,South,High Street
2,S03,Chennai,South,Mall
3,S04,Mumbai,West,High Street
4,S05,Pune,West,Mall


In [20]:
from pyspark.sql.functions import col, sum, avg, desc

**Exercises — Lazy Execution in PySpark**

**Part A — Basic transformations and actions**

1. Filter all transactions where quantity > 5 . Store the result in a new DataFrame without
displaying it.

In [23]:
sales_df = spark.read.csv("lazy_execution_sales_25000.csv", header=True, inferSchema=True)

# Apply the PySpark transformation
high_quantity_df = sales_df.filter(col("quantity") > 5)
print("Transformation has been created successfully using PySpark DataFrame!")

Transformation has been created successfully using PySpark DataFrame!


2. Select only order_id , product_id , quantity , and unit_price . Use show() to
trigger execution.

In [24]:
selected_df = sales_df.select("order_id", "product_id", "quantity", "unit_price")
selected_df.show()

+--------+----------+--------+----------+
|order_id|product_id|quantity|unit_price|
+--------+----------+--------+----------+
| O000001|      P002|       3|      2999|
| O000002|      P014|       6|     29999|
| O000003|      P008|       6|     22999|
| O000004|      P008|       7|     22999|
| O000005|      P007|       2|     79999|
| O000006|      P005|       8|      2499|
| O000007|      P010|       8|      3999|
| O000008|      P002|       9|      2999|
| O000009|      P007|       5|     79999|
| O000010|      P006|       5|     55999|
| O000011|      P007|       1|     79999|
| O000012|      P010|       9|      3999|
| O000013|      P013|       6|      5999|
| O000014|      P005|       8|      2499|
| O000015|      P007|       7|     79999|
| O000016|      P006|       1|     55999|
| O000017|      P004|       9|     16999|
| O000018|      P012|       7|      6999|
| O000019|      P014|       5|     29999|
| O000020|      P010|       7|      3999|
+--------+----------+--------+----

3. Filter transactions with unit_price > 20000 . Use count() to determine the number of
matching records.

In [25]:
expensive_count = sales_df.filter(col("unit_price") > 20000).count()
print("Number of expensive transactions:", expensive_count)

Number of expensive transactions: 6560


4. Create a new column gross_amount by multiplying quantity * unit_price . Inspect the
execution plan using explain() before displaying the results.

In [26]:
gross_df = sales_df.withColumn("gross_amount", col("quantity") * col("unit_price"))
gross_df.explain()
gross_df.show()

== Physical Plan ==
*(1) Project [order_id#102, order_date#103, store_id#104, product_id#105, quantity#106, unit_price#107, discount_pct#108, payment_method#109, order_status#110, sales_channel#111, (quantity#106 * unit_price#107) AS gross_amount#155]
+- FileScan csv [order_id#102,order_date#103,store_id#104,product_id#105,quantity#106,unit_price#107,discount_pct#108,payment_method#109,order_status#110,sales_channel#111] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...


+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|gross_amount|
+--------+----------+--------

5. Filter transactions with order_status = "Completed" and sort them by unit_price
descending. Trigger execution using take(10).

In [27]:
status_df = sales_df.filter(col("order_status") == "Completed").orderBy(desc("unit_price"))
status_df.take(10)

[Row(order_id='O000085', order_date=datetime.date(2026, 4, 22), store_id='S03', product_id='P007', quantity=4, unit_price=79999, discount_pct=0, payment_method='Card', order_status='Completed', sales_channel='Online'),
 Row(order_id='O000290', order_date=datetime.date(2026, 6, 26), store_id='S06', product_id='P007', quantity=3, unit_price=79999, discount_pct=10, payment_method='Net Banking', order_status='Completed', sales_channel='Online'),
 Row(order_id='O000098', order_date=datetime.date(2026, 4, 15), store_id='S05', product_id='P007', quantity=1, unit_price=79999, discount_pct=5, payment_method='Net Banking', order_status='Completed', sales_channel='Online'),
 Row(order_id='O000005', order_date=datetime.date(2026, 3, 14), store_id='S06', product_id='P007', quantity=2, unit_price=79999, discount_pct=0, payment_method='Cash', order_status='Completed', sales_channel='Online'),
 Row(order_id='O000107', order_date=datetime.date(2026, 5, 24), store_id='S05', product_id='P007', quantity=9

**Part B — Chained transformations**

6. Create a pipeline that filters completed orders, calculates gross amount, selects five relevant
columns, and sorts by gross amount descending. Use only one action at the end.

In [28]:
pipeline_df = sales_df.filter(col("order_status") == "Completed") \
                      .withColumn("gross_amount", col("quantity") * col("unit_price")) \
                      .select("order_id", "product_id", "quantity", "unit_price", "gross_amount") \
                      .orderBy(desc("gross_amount"))
pipeline_df.show()

+--------+----------+--------+----------+------------+
|order_id|product_id|quantity|unit_price|gross_amount|
+--------+----------+--------+----------+------------+
| O002603|      P007|       9|     79999|      719991|
| O006170|      P007|       9|     79999|      719991|
| O002693|      P007|       9|     79999|      719991|
| O000107|      P007|       9|     79999|      719991|
| O002841|      P007|       9|     79999|      719991|
| O000367|      P007|       9|     79999|      719991|
| O002940|      P007|       9|     79999|      719991|
| O000527|      P007|       9|     79999|      719991|
| O003108|      P007|       9|     79999|      719991|
| O000690|      P007|       9|     79999|      719991|
| O003542|      P007|       9|     79999|      719991|
| O001201|      P007|       9|     79999|      719991|
| O003884|      P007|       9|     79999|      719991|
| O001300|      P007|       9|     79999|      719991|
| O003897|      P007|       9|     79999|      719991|
| O001796|

7. Filter orders with discounts of at least 10%, calculate the discounted sales amount, and display
the top 15 orders.

In [30]:
discount_df = sales_df.filter(col("discount_pct") >= 10) \
                      .withColumn("discounted_sales", col("quantity") * col("unit_price") * (1 - col("discount_pct") / 100)) \
                      .orderBy(desc("discounted_sales"))
discount_df.show(15)

+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+----------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|discounted_sales|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+----------------+
| O006653|2026-06-29|     S01|      P007|       9|     79999|          10|          Card|   Cancelled|       Online|        647991.9|
| O011893|2026-03-26|     S07|      P007|       9|     79999|          10|           UPI|   Completed|       Online|        647991.9|
| O006699|2026-06-25|     S05|      P007|       9|     79999|          10|   Net Banking|   Completed|       Online|        647991.9|
| O003108|2026-02-23|     S08|      P007|       9|     79999|          10|   Net Banking|   Completed|     In Store|        647991.9|
| O007743|2026-05-22|     S08|      P007|       9|     79999| 

8. Group transactions by payment_method , calculate total quantity sold, and sort the results in
descending order.

In [31]:
sales_df.groupBy("payment_method") \
        .agg(sum("quantity").alias("total_quantity")) \
        .orderBy(desc("total_quantity")) \
        .show()

+--------------+--------------+
|payment_method|total_quantity|
+--------------+--------------+
|          Cash|         31754|
|          Card|         31327|
|           UPI|         30894|
|   Net Banking|         30889|
+--------------+--------------+



9. Group transactions by sales_channel and order_status , then calculate the transaction
count for each combination.

In [32]:
sales_df.groupBy("sales_channel", "order_status") \
        .count() \
        .show()

+-------------+------------+-----+
|sales_channel|order_status|count|
+-------------+------------+-----+
|       Online|   Completed|10797|
|     In Store|   Cancelled|  592|
|       Online|    Returned|  987|
|       Online|   Cancelled|  633|
|     In Store|    Returned| 1013|
|     In Store|   Completed|10978|
+-------------+------------+-----+



10. Create a DataFrame containing only order_id , store_id , product_id , and quantity .
Use drop() to remove store_id, then use first() to retrieve one record.


In [33]:
four_cols_df = sales_df.select("order_id", "store_id", "product_id", "quantity")
first_record = four_cols_df.drop("store_id").first()
print(first_record)

Row(order_id='O000001', product_id='P002', quantity=3)


**Part C — Joins and aggregations**

11. Join sales_df with products_df using product_id . Select product name, category,
quantity, and unit price. Display 20 records.

In [35]:
products_df = spark.read.csv("lazy_execution_products.csv", header=True, inferSchema=True)

# Perform the inner join and select the specified columns
sales_df.join(products_df, on="product_id", how="inner") \
        .select("product_name", "category", "quantity", "unit_price") \
        .show()

+-------------------+-----------+--------+----------+
|       product_name|   category|quantity|unit_price|
+-------------------+-----------+--------+----------+
|Mechanical Keyboard|Accessories|       3|      2999|
|      Smartphone 5G|    Mobiles|       6|     29999|
|     Android Tablet|    Tablets|       6|     22999|
|     Android Tablet|    Tablets|       7|     22999|
|          Laptop 16|  Computers|       2|     79999|
|          Webcam HD|Accessories|       8|      2499|
|  Bluetooth Speaker|      Audio|       8|      3999|
|Mechanical Keyboard|Accessories|       9|      2999|
|          Laptop 16|  Computers|       5|     79999|
|          Laptop 14|  Computers|       5|     55999|
|          Laptop 16|  Computers|       1|     79999|
|  Bluetooth Speaker|      Audio|       9|      3999|
|   External HDD 2TB|    Storage|       6|      5999|
|          Webcam HD|Accessories|       8|      2499|
|          Laptop 16|  Computers|       7|     79999|
|          Laptop 14|  Compu

12. Join sales_df with stores_df using store_id . Group by city and calculate the total
quantity sold.

In [37]:
stores_df = spark.read.csv("lazy_execution_stores.csv", header=True, inferSchema=True)

# Perform the inner join and group by city
sales_df.join(stores_df, on="store_id", how="inner") \
        .groupBy("city") \
        .agg(sum("quantity").alias("total_quantity")) \
        .show()

+---------+--------------+
|     city|total_quantity|
+---------+--------------+
|  Chennai|         15727|
|   Mumbai|         15489|
|  Kolkata|         15619|
|     Pune|         15407|
|    Delhi|         15101|
|Bengaluru|         15718|
|Hyderabad|         15784|
|   Jaipur|         16019|
+---------+--------------+



13. Join all three DataFrames. Calculate total completed-order revenue for each product category
and region.

In [40]:
full_df = sales_df.join(products_df, on="product_id", how="inner") \
                  .join(stores_df, on="store_id", how="inner")

full_df.filter(col("order_status") == "Completed") \
       .groupBy("category", "region") \
       .agg(sum(col("quantity") * col("unit_price")).alias("total_revenue")) \
       .show()

+-----------+------+-------------+
|   category|region|total_revenue|
+-----------+------+-------------+
|   Displays| North|     32468090|
|   Displays| South|     45761308|
|    Storage| South|     36976319|
|   Displays|  East|     13616199|
|      Audio|  East|     12043135|
|    Tablets| North|     38270336|
|    Mobiles| South|    120674524|
|    Storage|  East|     12140136|
|    Mobiles| North|     75482543|
|    Mobiles|  East|     41178126|
|Accessories|  East|      9093361|
|      Audio|  West|     25317111|
|    Tablets|  West|     37488370|
|   Displays|  West|     28286336|
|      Audio| South|     34629665|
|      Audio| North|     24327211|
|    Storage|  West|     23673341|
|  Computers| North|    240756425|
|  Computers|  West|    221092729|
|    Mobiles|  West|     80372388|
+-----------+------+-------------+
only showing top 20 rows


14. Filter completed transactions, join the product master, calculate discounted revenue, and display
the top five products by total revenue.

In [41]:
sales_df.filter(col("order_status") == "Completed") \
        .join(products_df, on="product_id", how="inner") \
        .withColumn("discounted_revenue", col("quantity") * col("unit_price") * (1 - col("discount_pct") / 100)) \
        .groupBy("product_name") \
        .agg(sum("discounted_revenue").alias("total_revenue")) \
        .orderBy(desc("total_revenue")) \
        .limit(5) \
        .show()

+---------------+--------------------+
|   product_name|       total_revenue|
+---------------+--------------------+
|      Laptop 16|5.2272546585000014E8|
|      Laptop 14| 3.830051604999999E8|
|  Smartphone 5G| 2.060406317500002E8|
| Android Tablet|1.5040656029999995E8|
|27-inch Monitor|1.1406074015000013E8|
+---------------+--------------------+



**Part D — Understanding lazy execution**

15. Create five consecutive transformations using filter() , select() , withColumn() ,
groupBy() , and orderBy() . Add a Python print() after each transformation. Observe
when Spark actually processes the data.

In [42]:
df1 = sales_df.filter(col("order_status") == "Completed")
print("Transformation 1 created")

df2 = df1.select("order_id", "product_id", "quantity", "unit_price")
print("Transformation 2 created")

df3 = df2.withColumn("gross_amount", col("quantity") * col("unit_price"))
print("Transformation 3 created")

df4 = df3.groupBy("product_id").agg(sum("gross_amount").alias("total_revenue"))
print("Transformation 4 created")

df5 = df4.orderBy(desc("total_revenue"))
print("Transformation 5 created")

df5.show()

Transformation 1 created
Transformation 2 created
Transformation 3 created
Transformation 4 created
Transformation 5 created
+----------+-------------+
|product_id|total_revenue|
+----------+-------------+
|      P007|    552313096|
|      P006|    404480777|
|      P014|    217222759|
|      P008|    158555106|
|      P004|    120131933|
|      P015|    100484822|
|      P011|     66268636|
|      P012|     52079559|
|      P013|     44860522|
|      P010|     30048486|
|      P002|     21319891|
|      P005|     18787482|
|      P009|     14302845|
|      P003|     13591143|
|      P001|      6046033|
+----------+-------------+



16. Create a filtered DataFrame and inspect it with explain() . Then call show() and count()
separately. Observe the Spark jobs in the Colab output.

In [44]:
filtered_df = sales_df.filter(col("quantity") > 5)
filtered_df.explain()
filtered_df.show()
print("Total records:", filtered_df.count())

== Physical Plan ==
*(1) Filter (isnotnull(quantity#106) AND (quantity#106 > 5))
+- FileScan csv [order_id#102,order_date#103,store_id#104,product_id#105,quantity#106,unit_price#107,discount_pct#108,payment_method#109,order_status#110,sales_channel#111] Batched: false, DataFilters: [isnotnull(quantity#106), (quantity#106 > 5)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000.csv], PartitionFilters: [], PushedFilters: [IsNotNull(quantity), GreaterThan(quantity,5)], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...


+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000002|2026-05-29| 

**Observation:** `explain()` creates no Spark job. `show()` and `count()` each trigger their own job(s) and re-run the plan from the source, because the DataFrame is not cached. You can call `filtered_df.cache()` before them to avoid recomputation (the first action fills the cache).
Check the Spark UI/job progress bars (`[Stage x:>...]`) printed under each cell.

17. Use collect() to retrieve the grouped total quantity for each product category. Explain why
collecting a small aggregated result is safer than collecting all 25,000 rows.

In [45]:
category = sales_df.join(products_df, on="product_id", how="inner").groupBy("category") \
                   .agg(sum("quantity").alias("total_quantity"))
category.collect()

[Row(category='Computers', total_quantity=16258),
 Row(category='Displays', total_quantity=7870),
 Row(category='Mobiles', total_quantity=16572),
 Row(category='Accessories', total_quantity=41839),
 Row(category='Storage', total_quantity=17251),
 Row(category='Audio', total_quantity=17159),
 Row(category='Tablets', total_quantity=7915)]

**Why collecting the aggregate is safer:** `collect()` pulls all results into the driver's memory as Python objects. The grouped result has only one row per category (a handful of rows), so it is tiny. Collecting all 25,000 raw rows is still modest here, but it moves the full dataset from the executors to a single machine, uses much more memory (Python `Row` objects are heavy), and doesn't scale. With millions of rows it can cause out-of-memory errors on the driver. Aggregate first (let Spark do the distributed work), then collect only the small result.

18. Filter completed orders, calculate discounted revenue, and save the resulting DataFrame as
Parquet using write.mode("overwrite").parquet() . Identify which operation triggers
execution.

In [47]:
result_df = sales_df.filter(col("order_status") == "Completed") \
                      .withColumn("discounted_revenue", col("quantity") * col("unit_price") * (1 - col("discount_pct") / 100))

print("Transformations defined (No execution triggered yet)")

# Action - This is what triggers the Spark job execution
result_df.write.mode("overwrite").parquet("completed_discounted_revenue.parquet")
print("Parquet file saved! (Execution triggered by the write.parquet() action)")

Transformations defined (No execution triggered yet)
Parquet file saved! (Execution triggered by the write.parquet() action)


**Caching**

In [1]:
from google.colab import files
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("SalesAnalysis").getOrCreate()

# Browse and select multiple CSV files
uploaded = files.upload()

# Load selected CSV files into PySpark DataFrames
dataframes = {}

for filename in uploaded.keys():
    df = spark.read.csv(filename, header=True, inferSchema=True)
    dataframes[filename] = df

    print(f"\nFile: {filename}")
    print("Records:", df.count())
    df.show(5)

Saving lazy_execution_stores (1).csv to lazy_execution_stores (1).csv
Saving lazy_execution_sales_25000 (1).csv to lazy_execution_sales_25000 (1).csv
Saving lazy_execution_products (1).csv to lazy_execution_products (1).csv

File: lazy_execution_stores (1).csv
Records: 8
+--------+---------+------+-----------+
|store_id|     city|region| store_type|
+--------+---------+------+-----------+
|     S01|Hyderabad| South|       Mall|
|     S02|Bengaluru| South|High Street|
|     S03|  Chennai| South|       Mall|
|     S04|   Mumbai|  West|High Street|
|     S05|     Pune|  West|       Mall|
+--------+---------+------+-----------+
only showing top 5 rows

File: lazy_execution_sales_25000 (1).csv
Records: 25000
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+---

In [2]:
from pyspark.sql import functions as F
from pyspark import StorageLevel
import time

# Find the sales dataset from the uploaded files
sales_df = None

for filename, df in dataframes.items():
    if {"order_id", "quantity", "unit_price", "order_status"}.issubset(df.columns):
        sales_df = df
        print("Sales file selected:", filename)
        break

if sales_df is None:
    raise ValueError("Sales CSV not found. Please upload the sales dataset.")

print("Total Sales Records:", sales_df.count())
sales_df.show(5)

Sales file selected: lazy_execution_sales_25000 (1).csv
Total Sales Records: 25000
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000005|2026-03-14|     S06|      P007|       2|     79999|           0|          Cash|   Comple

In [3]:
# Transformation
completed_df = sales_df.filter(F.col("order_status") == "Completed")
print("Transformation created")

# Action 1
print("Completed Orders:", completed_df.count())

# Action 2
completed_df.show(10)

Transformation created
Completed Orders: 21775
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000005|2026-03-14|     S06|      P007|       2|     79999|           0|          Cash|   Completed|       Online|
| O000006|2026-04

In [4]:
# Cache the filtered DataFrame
completed_df.cache()

print("Is Cached:", completed_df.is_cached)

# First action populates cache
completed_df.count()

# Reuse cached data
completed_df.show(10)

# Another action using cached data
completed_df.groupBy("product_id").count().show()

Is Cached: True
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000005|2026-03-14|     S06|      P007|       2|     79999|           0|          Cash|   Completed|       Online|
| O000006|2026-04-05|     S08|      P005|       

**Persist and Unpersist**

In [5]:
# Create filtered DataFrame
high_quantity_df = sales_df.filter(F.col("quantity") > 5)

# Persist using MEMORY_AND_DISK
high_quantity_df.persist(StorageLevel.MEMORY_AND_DISK)

# Trigger execution
high_quantity_df.count()

# Reuse persisted data
high_quantity_df.show(10)

# Check storage level
print("Storage Level:", high_quantity_df.storageLevel)

+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000006|2026-04-05|     S08|      P005|       8|      2499|           0|           UPI|   Completed|     In Store|
| O000007|2026-02-27|     S03|      P010|       8|      3999|          10|           UPI|   Completed|       Online|
| O000008|2026-03-21|     S07|      P002|       9|      2999|   

In [6]:
# Check cache status
print("Before:", completed_df.is_cached)

# Remove cache
completed_df.unpersist(blocking=True)

# Check again
print("After:", completed_df.is_cached)

# Remove the other persisted DataFrame
high_quantity_df.unpersist(blocking=True)

Before: True
After: False


DataFrame[order_id: string, order_date: date, store_id: string, product_id: string, quantity: int, unit_price: int, discount_pct: int, payment_method: string, order_status: string, sales_channel: string]

In [7]:
# Generate 40 batch IDs (0 to 39)
batch_df = spark.range(40).select(
    F.col("id").alias("batch_id")
)

batch_df.show(5)

# Combine each sales record with all 40 batch IDs
large_df = sales_df.crossJoin(batch_df)

print("Original Records:", sales_df.count())
print("Large Dataset Records:", large_df.count())

large_df.show(5)

+--------+
|batch_id|
+--------+
|       0|
|       1|
|       2|
|       3|
|       4|
+--------+
only showing top 5 rows
Original Records: 25000
Large Dataset Records: 1000000
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+--------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|batch_id|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+--------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|       0|
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|       1|
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|       2|
| O000001|2026-06-22|     S04|      P002|       3|      2999|     

In [8]:
# Create transformations
result_df = large_df.filter(
    F.col("quantity") > 3
).withColumn(
    "revenue",
    F.col("quantity") * F.col("unit_price")
).groupBy("product_id").agg(
    F.sum("revenue").alias("total_revenue")
)

# First execution
start = time.perf_counter()

result_df.collect()

time1 = time.perf_counter() - start

# Second execution
start = time.perf_counter()

result_df.collect()

time2 = time.perf_counter() - start

print("First Execution:", round(time1, 3), "seconds")
print("Second Execution:", round(time2, 3), "seconds")

First Execution: 1.385 seconds
Second Execution: 0.07 seconds


**Partitions**

In [9]:
# Check total records
print("Total Records:", sales_df.count())

# Check number of partitions
print("Number of Partitions:", sales_df.rdd.getNumPartitions())

Total Records: 25000
Number of Partitions: 1


In [10]:
# Create 8 partitions
partitioned_df = sales_df.repartition(8)

print("Original Partitions:", sales_df.rdd.getNumPartitions())
print("New Partitions:", partitioned_df.rdd.getNumPartitions())

# Trigger execution
partitioned_df.count()


Original Partitions: 1
New Partitions: 8


25000

In [11]:
from pyspark.sql import functions as F

# Add partition ID to every record
distribution_df = partitioned_df.withColumn(
    "partition_id",
    F.spark_partition_id()
)

# Count records in each partition
distribution_df.groupBy("partition_id") \
    .count() \
    .orderBy("partition_id") \
    .show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0| 3125|
|           1| 3125|
|           2| 3125|
|           3| 3125|
|           4| 3125|
|           5| 3125|
|           6| 3125|
|           7| 3125|
+------------+-----+



In [13]:
# Key-based Partition - Partition dada by store_id
store_partitioned_df = sales_df.repartition(8, "store_id")

print("Partitions:", store_partitioned_df.rdd.getNumPartitions())

# Inspect partition distribution
store_partitioned_df.withColumn("partition_id", F.spark_partition_id()) \
                  .groupBy("partition_id", "store_id") \
                  .count() \
                  .orderBy("partition_id", "store_id").show(30)

Partitions: 8
+------------+--------+-----+
|partition_id|store_id|count|
+------------+--------+-----+
|           0|     S01| 3180|
|           1|     S02| 3121|
|           1|     S08| 3122|
|           3|     S06| 3033|
|           4|     S05| 3117|
|           4|     S07| 3209|
|           5|     S03| 3132|
|           5|     S04| 3086|
+------------+--------+-----+



In [15]:
# Create 8 partitions
df8 = sales_df.repartition(8)

# Reduce using repartition
df_repartition = df8.repartition(3)

# Reduce using coalesce
df_coalesce = df8.coalesce(3)

print("Repartition:", df_repartition.rdd.getNumPartitions())
print("Coalesce:", df_coalesce.rdd.getNumPartitions())

# Examine execution plans
print("REPARTITION PLAN")
df_repartition.explain()

print("COALESCE PLAN")
df_coalesce.explain()

Repartition: 3
Coalesce: 3
REPARTITION PLAN
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 1
   +- ShuffleQueryStage 0
      +- Exchange RoundRobinPartitioning(3), REPARTITION_BY_NUM, [plan_id=1097]
         +- FileScan csv [order_id#64,order_date#65,store_id#66,product_id#67,quantity#68,unit_price#69,discount_pct#70,payment_method#71,order_status#72,sales_channel#73] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000 (1).csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...
+- == Initial Plan ==
   Exchange RoundRobinPartitioning(3), REPARTITION_BY_NUM, [plan_id=1097]
   +- FileScan csv [order_id#64,order_date#65,store_id#66,product_id#67,quantity#68,unit_price#69,discount_pct#70,payment_method#71,order_status#72,sales_channel#73] Batched: false, DataFilters: []

**Practice Exercises.**

1.Find the current number of partitions.

In [17]:
partitioned_df.count()

25000

2.Increase the number of partitions to 12.

In [18]:
# Create 12 partitions
new_partitioned_df = sales_df.repartition(12)

print("Original Partitions:", sales_df.rdd.getNumPartitions())
print("New Partitions:", new_partitioned_df.rdd.getNumPartitions())

Original Partitions: 1
New Partitions: 12


3.Display the number of records in each partition.

In [20]:
from pyspark.sql import functions as F

# Add partition ID to every record
distribution_df = new_partitioned_df.withColumn(
    "partition_id",
    F.spark_partition_id()
)

# Count records in each partition
distribution_df.groupBy("partition_id") \
    .count() \
    .orderBy("partition_id") \
    .show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0| 2084|
|           1| 2083|
|           2| 2083|
|           3| 2083|
|           4| 2083|
|           5| 2083|
|           6| 2083|
|           7| 2083|
|           8| 2083|
|           9| 2084|
|          10| 2084|
|          11| 2084|
+------------+-----+



4.Repartition the DataFrame into 6 partitions using product_id.

In [21]:
# Reduce using repartition
df_repartition = new_partitioned_df.repartition(6, "product_id")

print("Repartition:", df_repartition.rdd.getNumPartitions())

Repartition: 6


5.Reduce 12 partitions to 4 using coalesce().

In [22]:
# Reduce using coalesce
df_coalesce = new_partitioned_df.coalesce(4)

print("Coalesce:", df_coalesce.rdd.getNumPartitions())

Coalesce: 4
